# Project 3: how much did prices in Suriname rise?

**The question.** How much did prices in Suriname rise from 2010 to 2025, and what can we not say about 2018 and 2019?

**The data.** The World Bank's series of consumer price inflation for Suriname, in percent per year (indicator FP.CPI.TOTL.ZG). Each value is the change in the average cost of a basket of goods and services from the year before. The World Bank takes the series from the International Monetary Fund's International Financial Statistics and last updated it on 2026-10-08. The table was downloaded on 2026-11-06, and it has no value for 2018 or 2019.

**Your team.** Four roles rotate every week: builder, data checker, AI checker and presenter. This week the data checker makes sure that no number on the price sheet depends on a year the series does not have.

**Who it is for.** A community group in Suriname that supports older people living on a fixed pension. The group gets a price sheet of about 200 words that a volunteer can read aloud at a meeting.

**What you hand in on Friday.** This notebook with every CHECK passing, one chart, the price sheet for the community group, and your AI log.

## Load the data

This cell is given. It reads the table from the course folder, or straight from the World Bank if the file is not there. The years 2018 and 2019 are in the table as rows with an empty value.

In [ ]:
import json, urllib.request
import pandas as pd

URL = ("https://api.worldbank.org/v2/country/SUR/indicator/FP.CPI.TOTL.ZG"
       "?format=json&date=2010:2025&per_page=100")
try:
    inflation = pd.read_csv("../data/week03-inflation-2010-2025.csv")
except FileNotFoundError:
    raw = json.load(urllib.request.urlopen(URL))[1]
    inflation = pd.DataFrame(
        [{"year": int(r["date"]), "inflation_pct": r["value"]} for r in raw]
    ).sort_values("year").reset_index(drop=True)
inflation

## TODO 1: describe the table

Print the number of rows and the first and last year. Store in `missing_years` a list of the years that have no value.

In [ ]:
# TODO 1
missing_years = None

In [ ]:
# CHECK 1
expected = inflation.loc[inflation["inflation_pct"].isna(), "year"].tolist()
assert missing_years is not None, "missing_years is still empty"
assert sorted(int(y) for y in missing_years) == expected
print("CHECK 1 passed:", expected)

## TODO 2: chart inflation, gaps included

Draw a bar chart of inflation by year. Leave 2018 and 2019 empty, and label them on the chart as missing. Give the chart a title, labelled axes and the source.

In [ ]:
# TODO 2

## TODO 3: from percent to growth factor

Add a column `factor` to `inflation`: the number each year's prices were multiplied by, which is 1 plus the percentage divided by 100. The years 2018 and 2019 must stay empty.

In [ ]:
# TODO 3

In [ ]:
# CHECK 3
assert "factor" in inflation, "inflation has no factor column yet"
p = inflation.set_index("year")["inflation_pct"]
f = inflation.set_index("year")["factor"]
assert ((f - (1 + p / 100)).abs().dropna() < 1e-9).all(), "a rise of p percent multiplies prices by 1 + p/100"
assert f[p.isna()].isna().all(), "the missing years must stay empty, not 0 or 1"
print("CHECK 3 passed: the largest factor is", round(f.max(), 4), "in", int(f.idxmax()))

## TODO 4: the two stretches the data covers

The value for 2011 compares prices in 2011 with prices in 2010. Work out which years each of the two rises below uses before you write any code.

Store in `rise_2010_2017` how many times as high prices were in 2017 as in 2010, and in `rise_2019_2025` how many times as high they were in 2025 as in 2019. Print each one as a percentage rise too.

In [ ]:
# TODO 4
rise_2010_2017 = None
rise_2019_2025 = None

In [ ]:
# CHECK 4
g = 1 + inflation.set_index("year")["inflation_pct"] / 100
assert rise_2010_2017 is not None and rise_2019_2025 is not None, "one of the two rises is still empty"
assert rise_2010_2017 < 100, "store a multiple (2 means prices doubled), not a percentage"
assert abs(rise_2010_2017 - g.loc[2011:2017].prod()) < 1e-6, "the 2010 value compares 2010 with 2009, so it is not part of the rise from 2010"
assert abs(rise_2019_2025 - g.loc[2020:2025].prod()) < 1e-6, "the rise from 2019 starts with the 2020 value"
print("CHECK 4 passed:", round(rise_2010_2017, 2), "times and", round(rise_2019_2025, 2), "times")

## TODO 5: what the series cannot say

The rise from 2010 to 2025 needs every factor from 2011 to 2025, including 2018 and 2019. Store that rise in `total_2010_2025`, computed so that the answer is missing (`prod(skipna=False)`).

Then store in `at_least` the smallest that rise could have been if prices did not fall in 2018 or 2019, as a multiple of the 2010 price level. Write the assumption in a comment next to it.

In [ ]:
# TODO 5
total_2010_2025 = None
at_least = None

In [ ]:
# CHECK 5
import math
g = 1 + inflation.set_index("year")["inflation_pct"] / 100
assert total_2010_2025 is not None, "total_2010_2025 is still empty: it should be a missing value"
assert math.isnan(total_2010_2025), "this total needs 2018 and 2019, which the series does not have"
assert at_least is not None, "at_least is still empty"
assert abs(at_least - g.loc[2011:2017].prod() * g.loc[2020:2025].prod()) < 1e-6
print("CHECK 5 passed: the total is unknown, and it is at least", round(at_least, 2), "times if prices did not fall in 2018 or 2019")

## TODO 6: the AI check

Paste the table into your AI assistant and ask: *"Using this table of annual inflation, by how much did consumer prices in Suriname rise in total from 2010 to 2025?"* Write its answer, word for word, in your AI log.

Then compute three answers that a quiet mistake would give, each as a percentage rise from 2010 to 2025, and see whether the AI's answer is close to one of them.

- `if_added` adds the percentages for 2011 to 2025.
- `if_skipped` multiplies the factors for 2011 to 2025 and skips 2018 and 2019.
- `if_filled` multiplies the same factors after 2018 and 2019 are filled with a straight line between 2017 and 2020 (`interpolate()`).

Finally, ask the assistant without the table: *"What was inflation in Suriname in 2018 and in 2019?"* Record its numbers and the source it names. The World Bank series has no value for those years, so a number without a source you have opened does not go on your price sheet.

In [ ]:
# TODO 6
if_added = None
if_skipped = None
if_filled = None

In [ ]:
# CHECK 6
p = inflation.set_index("year")["inflation_pct"]
g = 1 + p / 100
assert None not in (if_added, if_skipped, if_filled), "one of the three is still empty"
assert abs(if_added - p.loc[2011:2025].sum()) < 1e-6, "if_added: add the percentages for 2011 to 2025"
assert abs(if_skipped - (g.loc[2011:2025].prod() - 1) * 100) < 1e-6, "if_skipped: a percentage rise, with 2018 and 2019 skipped"
assert abs(if_filled - ((1 + p.interpolate() / 100).loc[2011:2025].prod() - 1) * 100) < 1e-6, "if_filled: fill with interpolate() first"
print("CHECK 6 passed:", round(if_added, 1), round(if_skipped, 1), round(if_filled, 1))

## TODO 7: the price sheet for the community group

In a text cell below, write about 200 words that a volunteer could read aloud to older people living on a fixed pension. Give the rise for each stretch the data covers, and say what a sum of money that stayed the same since 2019 buys in 2025, as a share of what it bought in 2019. Say plainly that this series cannot give the total from 2010 to 2025, and why. Give the lowest the total could be, with the assumption behind it, and name the source. Use plain words and no code.